In [1]:
import os
import mne
import json
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

from collections import Counter

import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report, f1_score
from sklearn.model_selection import StratifiedKFold, KFold, train_test_split

from scipy import stats
from scipy.stats import ttest_rel
# from scipy.stats import false_discovery_control

# Read data

In [2]:
data_folder = '/mnt/s3-data2/amiftakhova/OCD/OCD/'
folders = [x for x in os.listdir(data_folder) if os.path.isdir(data_folder + x)]
inside_folders = []
for fldr in folders:
    ins = [fldr + '/' + x for x in os.listdir(data_folder + fldr) if os.path.isdir(data_folder + fldr + '/' + x)]
    if len(ins) > 0:
        inside_folders.extend(ins)
    else:
        inside_folders.append(fldr)
fldr2label = {inside_folders[i]: i for i in range(len(inside_folders))}
fldr2label

{'Response to stress (57)/Somatoform_disorder (12)': 0,
 'Response to stress (57)/Stress_disorder (24)': 1,
 'Response to stress (57)/eating_disorder (14)': 2,
 'Response to stress (57)/neurasthenia (7)': 3,
 'anxiety_disorder (81)/ADD (47)': 4,
 'anxiety_disorder (81)/AFD (18)': 5,
 'anxiety_disorder (81)/general_AD (16)': 6,
 'bipolar_disorder (61)/BPD_1(26)': 7,
 'bipolar_disorder (61)/BPD_2(25)': 8,
 'bipolar_disorder (61)/Cyclothymia(10)': 9,
 'controls (157)': 10,
 'depression (109)/mild (29)': 11,
 'depression (109)/moderade (47)': 12,
 'depression (109)/severe (32)': 13,
 'personality_disorder (56)': 14}

In [3]:
label2class = {
    4: 0, 5: 0, 6: 0, # anxiety
    7: 1, 8: 1, 9: 1, # bipolar
    10: 2, # control
    11: 3, 12: 3, 13: 3, # depression
    14: 4, # personality disorder,
    0: 5, 1: 5, 2: 5, 3: 5# stress
}

class2name = {0: 'anxiety', 1: 'bipolar', 2: 'control', 3: 'depression', 4: 'personality disorder', 5: 'stress'}
name2class = {v: k for k, v in class2name.items()}

In [4]:
og_files = []
zg_files = []
og_labels = []
zg_labels = []
for fldr in inside_folders:
    pth =  data_folder + fldr
    og_pths = [x for x in os.listdir(pth) if x.lower().endswith('.edf') and ('og.' in x.lower() or '_ог.' in x.lower() or ' ог.' in x.lower() or 'eo.' in x.lower() or '_eo' in x.lower() or '_of' in x.lower())]
    zg_pths = [x for x in os.listdir(pth) if x.lower().endswith('.edf') and ('zg.' in x.lower() or 'зог.' in x.lower() or 'зг.' in x.lower() or 'ec.' in x.lower() or 'fon.' in x.lower() or '_ec' in x.lower() or 'eс.' in x.lower())]
    left = [x for x in os.listdir(pth) if x not in og_pths and x not in zg_pths]
    if len(left) > 0:
        print(pth, left)
    for f in og_pths:
        og_files.append(pth + '/' + f)
        og_labels.append(fldr2label[fldr])
    for f in zg_pths:
        zg_files.append(pth + '/' + f)
        zg_labels.append(fldr2label[fldr])

print(f'Number of files for open eyes: {len(og_files)}')
print(f'Number of files for closed eyes: {len(zg_files)}')

# ensured that og_files[i] is the pair for zg_files[i]

Number of files for open eyes: 518
Number of files for closed eyes: 518


In [5]:
lens = []
for f in tqdm(og_files):
    try:
        sample = mne.io.read_raw_edf(f, verbose=False)
        lens.append(1.0 * len(sample) / sample.info['sfreq'])
    except Exception as e:
        print(f, e)

short_recordings = []
for i in tqdm(range(len(og_files))):
    og_file = og_files[i]
    zg_file = zg_files[i]
    try:
        og_sample = mne.io.read_raw_edf(og_file, verbose=False)
        og_len = 1.0 * len(og_sample) / sample.info['sfreq']
        zg_sample = mne.io.read_raw_edf(zg_file, verbose=False)
        zg_len = 1.0 * len(zg_sample) / zg_sample.info['sfreq']
        if zg_len < 20 or og_len < 20:
            short_recordings.append(i)
    except Exception as e:
        print(f, e)

og_files = [x for i, x in enumerate(og_files) if i not in short_recordings]
zg_files = [x for i, x in enumerate(zg_files) if i not in short_recordings]
og_labels = [x for i, x in enumerate(og_labels) if i not in short_recordings]
zg_labels = [x for i, x in enumerate(zg_labels) if i not in short_recordings]

og_no_stress = [x for i, x in enumerate(og_files) if label2class[og_labels[i]] != 5]
zg_no_stress = [x for i, x in enumerate(zg_files) if label2class[zg_labels[i]] != 5]
og_labels_no_stress = [x for i, x in enumerate(og_labels) if label2class[og_labels[i]] != 5]
zg_labels_no_stress = [x for i, x in enumerate(zg_labels) if label2class[zg_labels[i]] != 5]

  0%|          | 0/518 [00:00<?, ?it/s]

/mnt/s3-data2/amiftakhova/OCD/OCD/controls (157)/Kutuz_f23_contr_og.edf could not convert string to float: ''
/mnt/s3-data2/amiftakhova/OCD/OCD/controls (157)/Skopincev_20_EO_free.edf could not convert string to float: ''


  0%|          | 0/518 [00:00<?, ?it/s]

/mnt/s3-data2/amiftakhova/OCD/OCD/personality_disorder (56)/ИОДКОВСКАЯ ОЛЬГА 14 Ф60-3_og.EDF could not convert string to float: ''
/mnt/s3-data2/amiftakhova/OCD/OCD/personality_disorder (56)/ИОДКОВСКАЯ ОЛЬГА 14 Ф60-3_og.EDF could not convert string to float: ''


# Extract microstates info

In [6]:
from pycrostates.cluster import ModKMeans
from pycrostates.preprocessing import resample, apply_spatial_filter

In [ ]:
class_labels = [name2class[x] for x in og_labels]
train_ids, test_ids = train_test_split([i for i in range(len(og_files))], test_size=0.2, stratify=[i for i in class_labels], random_state=92)

In [9]:
s_freq = 250
channels2use = ['Fp1', 'Fp2', 'F3', 'Fz', 'F4', 'F7', 'F8', 'T3', 'T4', 'C3', 'Cz', 'C4', 'T5', 'T6', 'P3', 'Pz', 'P4', 'O1', 'O2']
to_skip = ['BORUTTO_JANNA_VLADIMIROVNA', 'Kutuz_f23_contr', 'MANUILOVA_ELENA_55', 'Martinenko_m45', 'Skopincev_20', 'FiAV_m50']

train_recordings = []
train_labels = []
for i in tqdm(train_ids):
    path = zg_files[ids[i]]
    if any([x in path for x in to_skip]):
        continue
    sample = mne.io.read_raw_edf(path, verbose=False, preload=True)
    sample = sample.resample(s_freq, verbose=False)

    sample = sample.filter(l_freq=1, h_freq=30, method='iir', verbose=False)
    channels = sample.ch_names
    to_drop = channels[19:]

    new_idx = []
    skip = False
    for ch in channels2use:
        found = False
        for k in range(19):
            if ch in channels[k]:
                new_idx.append(k)
                found = True
                break
        if not found:
            skip = True
            break
    sample = sample.pick(np.array(channels)[new_idx])
    sample = sample.crop(tmin=0.0, tmax=20.0)
    sample = sample.rename_channels({channels[new_idx[i]]: channels2use[i] for i in range(len(channels2use))})
    train_recordings.append(sample)
    train_labels.append(class_labels[i])

  0%|          | 0/412 [00:00<?, ?it/s]

In [10]:
concat_train_recordings = mne.concatenate_raws(train_recordings)

In [11]:
n_clusters = 4
ModK = ModKMeans(n_clusters=n_clusters, random_state=42)
ModK.fit(concat_train_recordings, n_jobs=-1)
microstate_maps = ModK.cluster_centers_

In [12]:
microstate_maps

array([[-3.07576397e-01, -3.16806343e-01, -3.19786149e-01,
        -3.46220003e-01, -3.24760568e-01, -2.38359505e-01,
        -2.40886453e-01, -1.33950806e-01, -1.31019976e-01,
        -2.68624852e-01, -3.22371843e-01, -2.62851479e-01,
        -2.70671371e-03,  2.11083737e-02, -1.49628014e-01,
        -1.99350232e-01, -1.38868633e-01, -1.23971015e-03,
        -2.09097135e-02],
       [-1.73652995e-04,  6.67566794e-04, -3.13692116e-04,
         2.41059086e-05,  6.25331462e-05, -7.50923892e-05,
         4.61898858e-04, -2.25189637e-04, -9.99999077e-01,
        -4.67998306e-05, -3.19229616e-04,  4.95247584e-04,
         5.00724076e-05,  2.69737264e-04, -2.06334919e-04,
        -5.33504725e-05,  4.39034578e-04, -2.69272827e-05,
         5.77408677e-04],
       [ 2.51556381e-01,  3.22949180e-01, -3.49138621e-02,
         2.72104074e-01, -1.04273954e-01,  2.61347869e-01,
         2.71160833e-01,  1.92644870e-01,  1.92847557e-01,
         2.54581173e-01,  2.43117510e-01,  2.40284968e-01,
    

In [19]:
import pandas as pd

def compute_durations(labels, sfreq):
    """Compute durations (in seconds) of each microstate segment."""
    durations = []
    current_label = labels[0]
    current_duration = 1 / sfreq
    
    for label in labels[1:]:
        if label == current_label:
            current_duration += 1 / sfreq
        else:
            durations.append((current_label, current_duration))
            current_label = label
            current_duration = 1 / sfreq
    durations.append((current_label, current_duration))
    return durations

def extract_features(durations):
    """Convert durations into a feature vector (mean, total time, etc.)."""
    df = pd.DataFrame(durations, columns=['Microstate', 'Duration'])
    
    features = {}
    for ms in range(n_clusters):
        ms_durations = df[df['Microstate'] == ms]['Duration']
        features[f'std_dur_ms{ms}'] = ms_durations.std()
        features[f'mean_dur_ms{ms}'] = ms_durations.mean()
        features[f'total_time_ms{ms}'] = ms_durations.sum()
        features[f'n_occurrences_ms{ms}'] = len(ms_durations)
    
    return features

In [20]:
train_features = []

for raw in train_recordings:
    # Predict microstates using the group-level model
    labels = ModK.predict(raw)
    
    # Compute features (durations, transitions, etc.)
    durations = compute_durations(labels.labels, sfreq=s_freq)
    features = extract_features(durations)
    train_features.append(features)

features_df = pd.DataFrame(train_features)
features_df['class'] = train_labels
features_df.to_csv('train_features.csv', index=False)

In [25]:
test_ids

[201,
 75,
 472,
 328,
 349,
 234,
 279,
 480,
 355,
 210,
 154,
 114,
 172,
 249,
 117,
 336,
 500,
 393,
 490,
 40,
 30,
 337,
 286,
 151,
 423,
 499,
 185,
 402,
 26,
 347,
 150,
 115,
 416,
 51,
 72,
 368,
 433,
 304,
 352,
 6,
 374,
 103,
 311,
 509,
 351,
 435,
 483,
 84,
 253,
 85,
 461,
 397,
 350,
 406,
 127,
 306,
 68,
 171,
 494,
 246,
 410,
 212,
 342,
 412,
 23,
 503,
 99,
 268,
 213,
 250,
 91,
 510,
 180,
 175,
 288,
 353,
 280,
 86,
 18,
 145,
 130,
 453,
 357,
 169,
 157,
 198,
 331,
 39,
 343,
 380,
 371,
 434,
 124,
 57,
 10,
 372,
 38,
 15,
 460,
 275,
 403,
 459,
 238]

In [26]:
s_freq = 250
channels2use = ['Fp1', 'Fp2', 'F3', 'Fz', 'F4', 'F7', 'F8', 'T3', 'T4', 'C3', 'Cz', 'C4', 'T5', 'T6', 'P3', 'Pz', 'P4', 'O1', 'O2']
to_skip = ['BORUTTO_JANNA_VLADIMIROVNA', 'Kutuz_f23_contr', 'MANUILOVA_ELENA_55', 'Martinenko_m45', 'Skopincev_20', 'FiAV_m50']

test_labels = []
test_features = []
for i in tqdm(test_ids):
    path = zg_files[ids[i]]
    if any([x in path for x in to_skip]):
        continue
    sample = mne.io.read_raw_edf(path, verbose=False, preload=True)
    sample = sample.resample(s_freq, verbose=False)

    sample = sample.filter(l_freq=1, h_freq=30, method='iir', verbose=False)
    channels = sample.ch_names
    to_drop = channels[19:]

    new_idx = []
    skip = False
    for ch in channels2use:
        found = False
        for k in range(19):
            if ch in channels[k]:
                new_idx.append(k)
                found = True
                break
        if not found:
            skip = True
            break
    sample = sample.pick(np.array(channels)[new_idx])
    sample = sample.crop(tmin=0.0, tmax=20.0)
    sample = sample.rename_channels({channels[new_idx[i]]: channels2use[i] for i in range(len(channels2use))})
    test_labels.append(class_labels[i])

    labels = ModK.predict(sample)
    
    # Compute features (durations, transitions, etc.)
    durations = compute_durations(labels.labels, sfreq=s_freq)
    features = extract_features(durations)
    test_features.append(features)

features_df = pd.DataFrame(test_features)
features_df['class'] = test_labels
features_df.to_csv('test_features.csv', index=False)

  0%|          | 0/103 [00:00<?, ?it/s]

In [27]:
train_features = pd.read_csv('train_features.csv')
test_features = pd.read_csv('test_features.csv')

In [28]:
test_features

,std_dur_ms0,mean_dur_ms0,total_time_ms0,n_occurrences_ms0,std_dur_ms1,mean_dur_ms1,total_time_ms1,n_occurrences_ms1,std_dur_ms2,mean_dur_ms2,total_time_ms2,n_occurrences_ms2,std_dur_ms3,mean_dur_ms3,total_time_ms3,n_occurrences_ms3,class
0,0.015731,0.021708,5.644,260,0.010090,0.011446,1.900,166,0.007201,0.009091,0.900,99,0.021973,0.037013,11.548,312,2
1,0.010873,0.015360,6.236,406,0.004147,0.006458,1.072,166,0.005156,0.007306,0.884,121,0.014720,0.025524,11.792,462,0
2,0.015533,0.022510,9.184,408,0.005231,0.007614,1.500,197,0.005255,0.007704,1.040,135,0.015133,0.021277,8.128,382,4
3,0.013804,0.021854,9.004,412,0.003375,0.005736,0.304,53,0.004921,0.007861,1.580,201,0.015083,0.022657,9.108,402,2
4,0.015598,0.023536,9.532,405,0.002947,0.005568,0.412,74,0.005325,0.008021,1.524,190,0.012573,0.020691,8.504,411,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
98,0.011557,0.016502,6.964,422,0.004990,0.007426,1.500,202,0.005667,0.009159,2.308,252,0.014216,0.020589,9.224,448,3
99,0.016635,0.031408,10.616,338,0.007622,0.008000,0.832,104,0.006471,0.008557,1.044,122,0.016381,0.023498,7.496,319,2
100,0.015108,0.021140,7.568,358,0.004145,0.006676,0.948,142,0.007142,0.009614,1.596,166,0.019298,0.026255,9.872,376,3
101,0.007059,0.010979,3.744,341,0.004237,0.006769,1.408,208,0.004183,0.007369,1.496,203,0.011283,0.029257,13.312,455,3


In [31]:
from sklearn.svm import SVC

X_train = train_features.drop('class', axis=1)
y_train = train_features['class']
X_test = test_features.drop('class', axis=1)
y_test = test_features['class']

clf = SVC(kernel='linear', random_state=92)
clf.fit(X_train, y_train)
preds = clf.predict(X_test)
print('F1-score: ', f1_score(y_test, preds, average='macro'))

F1-score:  0.10777343287119327


In [32]:
from sklearn.ensemble import RandomForestClassifier

X_train = train_features.drop('class', axis=1)
y_train = train_features['class']
X_test = test_features.drop('class', axis=1)
y_test = test_features['class']

clf = RandomForestClassifier()
clf.fit(X_train, y_train)
preds = clf.predict(X_test)
print('F1-score: ', f1_score(y_test, preds, average='macro'))

F1-score:  0.28275144875764074


In [34]:
from sklearn.neighbors import KNeighborsClassifier

X_train = train_features.drop('class', axis=1)
y_train = train_features['class']
X_test = test_features.drop('class', axis=1)
y_test = test_features['class']

clf = KNeighborsClassifier(weights='distance')
clf.fit(X_train, y_train)
preds = clf.predict(X_test)
print('F1-score: ', f1_score(y_test, preds, average='macro'))

F1-score:  0.30358736942070275
